# Fitts Law & Bewegungsanalyse – Thumb Reach

Auswertung der Ausgabe von `evaluate_fitts.py`:
- **Fitts Law**: MT = a + b·ID (Movement Time vs. Index of Difficulty)
- **Min-Jerk**: Integral des quadrierten Jerks
- **Geschwindigkeitsprofil**: Ballistic vs. Refinement Phase

## 1. Daten laden

`EVAL_DIR` auf den Ausgabeordner von `evaluate_fitts.py` setzen (z.B. `logs/<run>/eval_fitts`). Alle Diagramme werden dort als PNG abgelegt (`fitts_mt_vs_id.png`, `velocity_profiles.png`, `trajectories_overview.png`).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy import stats
from pathlib import Path

# Eval-Output-Ordner (anpassen)
EVAL_DIR = "logs/Constrained_Soft_13_seed1/eval_fitts"
os.makedirs(EVAL_DIR, exist_ok=True)
trials_path = os.path.join(EVAL_DIR, "trials.csv")
traj_dir = os.path.join(EVAL_DIR, "trajectories")

df = pd.read_csv(trials_path)
print(df.head())
print("\nSpalten:", df.columns.tolist())
print("Anzahl Trials:", len(df))

## 2. Fitts Law: MT vs. ID

Lineare Regression MT = a + b·ID. Erwartung: linearer Anstieg.

In [ ]:
# Nur erfolgreiche Trials für Fitts (optional: alle verwenden)
df_success = df[df["success"] == True].copy()
if len(df_success) < 2:
    df_success = df.copy()
    print("Warnung: Wenige erfolgreiche Trials, nutze alle.")

x = df_success["ID"].values
y = df_success["MT"].values
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.scatter(df["ID"], df["MT"], alpha=0.6, label="Trials")
ax.scatter(df_success["ID"], df_success["MT"], alpha=0.8, label="Erfolgreich")
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line, "k-", label=f"MT = {intercept:.3f} + {slope:.3f}·ID")
ax.set_xlabel("Index of Difficulty (ID)")
ax.set_ylabel("Movement Time (s)")
ax.set_title(f"Fitts Law (R² = {r_value**2:.3f}, p = {p_value:.4f})")
ax.legend()
plt.tight_layout()
_fig_fitts = os.path.join(EVAL_DIR, "fitts_mt_vs_id.png")
plt.savefig(_fig_fitts, dpi=150, bbox_inches="tight")
print(f"Gespeichert: {_fig_fitts}")
plt.show()

## 3. Geschwindigkeitsprofil & Ballistic / Refinement

Pro Trajektorie: Geschwindigkeit v(t) = ‖Δposition/Δt‖. Ballistic Phase: bis zum ersten Maximum; Refinement: danach.

In [ ]:
STEP_DT = 0.02  # Sekunden pro Step (wie in evaluate_fitts.py)

def velocity_from_trajectory(t, position):
    """position: (N, 2); returns speed (N-1,) and mid-times (N-1,)."""
    d = np.diff(position, axis=0)
    speed = np.linalg.norm(d, axis=1) / STEP_DT
    t_mid = (t[:-1] + t[1:]) / 2
    return speed, t_mid

def strongest_accel_time(t_mid, speed):
    """Zeitpunkt des stärksten positiven Geschwindigkeitssprungs (max Δv)."""
    if len(speed) < 3:
        return np.nan
    d_speed = np.diff(speed)
    j = int(np.argmax(d_speed))
    if d_speed[j] <= 0:
        return np.nan
    # Δv_j ist zwischen speed[j] und speed[j+1] -> markiere t_{j+1}
    return t_mid[j + 1]

def trim_autoreset_tail(pos, start_pos):
    """Entfernt ggf. einen VecEnv-Autoreset-Sprung am Trajektorien-Ende."""
    pos = np.asarray(pos, dtype=float)
    sp = np.asarray(start_pos, dtype=float).ravel()[:2]
    while len(pos) >= 2:
        last, prev = pos[-1, :2], pos[-2, :2]
        d_ls = float(np.linalg.norm(last - sp))
        d_ps = float(np.linalg.norm(prev - sp))
        jump = float(np.linalg.norm(last - prev))
        if jump >= 0.06 and d_ls <= 0.2 and d_ps > d_ls + 0.04:
            pos = pos[:-1]
        else:
            break
    return pos

# Figure 1: Beispiel-Episoden
n_plot = min(5, len(df))
fig, ax = plt.subplots(1, 1, figsize=(8, 4))

for ep in range(n_plot):
    npz_path = os.path.join(traj_dir, f"ep_{ep:04d}.npz")
    if not os.path.isfile(npz_path):
        continue
    data = np.load(npz_path)
    t = np.asarray(data["t"], dtype=float)
    pos = np.asarray(data["position"], dtype=float)
    if "start_pos" in data.files:
        pos = trim_autoreset_tail(pos, data["start_pos"])
    if len(pos) < 2:
        continue
    # Zeitreihe passend zur ggf. getrimmten Position kürzen
    t = t[: len(pos)]
    speed, t_mid = velocity_from_trajectory(t, pos)
    ax.plot(t_mid, speed, alpha=0.7, label=f"Ep {ep}")
    t_acc = strongest_accel_time(t_mid, speed)
    if not np.isnan(t_acc):
        ax.axvline(t_acc, color="gray", alpha=0.5, linestyle="--")

ax.set_xlabel("Time (s)")
ax.set_ylabel("Velocity (norm/s)")
ax.set_title("Velocity Profiles (Example Episodes, dashed = max Δv)")
ax.legend()
plt.tight_layout()
_fig_vel_examples = os.path.join(EVAL_DIR, "velocity_profiles_examples.png")
plt.savefig(_fig_vel_examples, dpi=150, bbox_inches="tight")
print(f"Gespeichert: {_fig_vel_examples}")
plt.show()

# Figure 2: pro Target genau eine Episode
fig, (ax, ax_leg) = plt.subplots(
    1, 2, figsize=(10.5, 5), gridspec_kw={"width_ratios": [1.0, 0.36]}
)
for target_idx in sorted(df["target_idx"].unique()):
    c = f"C{int(target_idx) % 10}"
    subset = df[df["target_idx"] == target_idx]
    for ep in subset["episode_id"].values:
        npz_path = os.path.join(traj_dir, f"ep_{int(ep):04d}.npz")
        if not os.path.isfile(npz_path):
            continue
        data = np.load(npz_path)
        t = np.asarray(data["t"], dtype=float)
        pos = np.asarray(data["position"], dtype=float)
        if "start_pos" in data.files:
            pos = trim_autoreset_tail(pos, data["start_pos"])
        if len(pos) < 2:
            continue
        t = t[: len(pos)]
        speed, t_mid = velocity_from_trajectory(t, pos)
        ax.plot(
            t_mid, speed, color=c, alpha=0.55, lw=1.2,
            label=f"Target {int(target_idx) + 1}",
        )
        # Kreuz am Moment des groessten positiven Geschwindigkeitssprungs
        if len(speed) >= 3:
            d_speed = np.diff(speed)
            j = int(np.argmax(d_speed))
            if d_speed[j] > 0:
                ax.plot(t_mid[j + 1], speed[j + 1], "+", color=c, ms=10, mew=2.0, label="_nolegend_")
        break

ax.set_xlabel("Time (s)")
ax.set_ylabel("Velocity (norm/s)")
ax.set_title("Velocity Profiles per Target")
# Ein allgemeiner Legenden-Eintrag fuer das Kreuz (nicht pro Target)
ax.plot([], [], "+", color="black", ms=10, mew=2.0, label="moment of greatest\nacceleration")

# Rechte Legenden-Box über volle Höhe
handles, labels = ax.get_legend_handles_labels()
ax_leg.set_xticks([])
ax_leg.set_yticks([])
for s in ax_leg.spines.values():
    s.set_visible(True)
ax_leg.legend(handles, labels, loc="upper left", fontsize=7, frameon=False)

plt.tight_layout()
_fig_vel_target = os.path.join(EVAL_DIR, "velocity_profiles_per_target.png")
plt.savefig(_fig_vel_target, dpi=150, bbox_inches="tight")
print(f"Gespeichert: {_fig_vel_target}")
plt.show()

## 4. Min-Jerk

Jerk = d³position/dt³. Metrik: Integral des quadrierten Jerks über die Bewegung (je kleiner, desto „glatter“).

In [ ]:
def jerk_from_position(t, position, dt=STEP_DT):
    """position (N, 2) -> jerk (N-3,) via finite differences."""
    # d³x/dt³ ≈ (x[i+2] - 2*x[i+1] + 2*x[i-1] - x[i-2]) / (2*dt^3)
    p = np.asarray(position)
    if len(p) < 5:
        return np.array([]), np.array([])
    jerk = (p[4:] - 2*p[3:-1] + 2*p[1:-3] - p[:-4]) / (2 * dt**3)
    jerk_norm = np.linalg.norm(jerk, axis=1)
    t_j = t[2:-2]
    return jerk_norm, t_j

def integral_squared_jerk(t, position, dt=STEP_DT):
    """Integral of squared jerk (Min-Jerk-Metrik)."""
    j, t_j = jerk_from_position(t, position, dt)
    if len(j) == 0:
        return np.nan
    return np.trapz(j**2, t_j)

min_jerk_values = []
for ep in range(len(df)):
    npz_path = os.path.join(traj_dir, f"ep_{ep:04d}.npz")
    if not os.path.isfile(npz_path):
        min_jerk_values.append(np.nan)
        continue
    data = np.load(npz_path)
    ij = integral_squared_jerk(data["t"], data["position"])
    min_jerk_values.append(ij)

df["integral_squared_jerk"] = min_jerk_values
print(df[["episode_id", "target_idx", "MT", "success", "integral_squared_jerk"]].head(10))
print("\nMittelwert Integral Squared Jerk (erfolgreich):", df.loc[df["success"], "integral_squared_jerk"].mean())

## 5. Positions-Trajektorien (Eval)

Eine repräsentative Episode pro Ziel aus `trajectories/ep_XXXX.npz`. Nach `terminated` kann der VecEnv schon reset sein — dann wäre der **letzte Punkt** wieder nahe Start und die Linie wirkt **geschlossen**; im Eval-Skript und optional hier per `TRAJ_TRIM_AUTORESET_TAIL` wird dieser Schwanz entfernt. **Sterne** = Ist-Ende der Bahn. **Plus (+)** = Joystick-Position am **größten Sprung der Schrittgeschwindigkeit** \(v_k=\|p_{k+1}-p_k\|/\Delta t\) (aus den gespeicherten Punkten berechnet; nicht extra in der npz abgelegt).

In [ ]:
# Filter: nur Trials mit gewünschtem Radius bzw. W (Fitts: W = 2 * target_radius im norm. Raum)
# Beides None = alle Radien. Wenn beides gesetzt ist, müssen BEIDE passen.
TRAJ_TARGET_RADIUS = None  # z.B. 0.2
TRAJ_TARGET_W = None       # z.B. 0.4  (entspricht radius 0.2)
RTOL, ATOL = 1e-9, 1e-6
# Alte npz: VecEnv kann nach Episode-Ende einen extra Punkt nahe Start speichern → geschlossene Schleife
TRAJ_TRIM_AUTORESET_TAIL = True
# Optional: Pfad nur bis zum ersten Treffer (inkl.); wirkt zusätzlich zur Obigen
TRAJ_CLIP_AT_FIRST_HIT = False
# Kreuz (+) an Joystick-Position mit groesstem Geschwindigkeitssprung v_k = ||p_{k+1}-p_k||/dt
STEP_DT = 0.02
TRAJ_MARK_ACCEL_SPIKE = True

df_traj = df.copy()
mask = np.ones(len(df_traj), dtype=bool)
if TRAJ_TARGET_RADIUS is not None:
    mask &= np.isclose(df_traj["target_radius"].values, float(TRAJ_TARGET_RADIUS), rtol=RTOL, atol=ATOL)
if TRAJ_TARGET_W is not None:
    if "W" not in df_traj.columns:
        raise KeyError("trials.csv hat keine Spalte 'W' — älteres Eval?")
    mask &= np.isclose(df_traj["W"].values, float(TRAJ_TARGET_W), rtol=RTOL, atol=ATOL)
df_traj = df_traj.loc[mask]
print(
    "Trajektorien-Filter:",
    f"radius={TRAJ_TARGET_RADIUS}", f"W={TRAJ_TARGET_W}",
    f"-> {len(df_traj)} / {len(df)} Trials",
)
if len(df_traj) == 0:
    raise ValueError("Keine Trials nach Radius/W-Filter — Werte anpassen oder None setzen.")
print("Vorkommende Radien:", sorted(df["target_radius"].unique()))
if "W" in df.columns:
    print("Vorkommende W:", sorted(df["W"].unique()))

# Genau eine repraesentative Trajektorie pro Target (max. 20 Linien)
fig, (ax, ax_leg) = plt.subplots(
    1, 2, figsize=(10.0, 7.5), gridspec_kw={"width_ratios": [1.0, 0.36]}
)

n_drawn = 0
for tid in sorted(df_traj["target_idx"].unique()):
    subset = df_traj[df_traj["target_idx"] == tid].copy()
    if len(subset) == 0:
        continue

    # Erfolgreiche bevorzugen; falls keine erfolgreichen vorhanden, alle nutzen
    subset_ok = subset[subset["success"] == True]
    subset_pick = subset_ok if len(subset_ok) > 0 else subset

    # Repräsentative Episode: die mit MT am naechsten zur medianen MT dieses Targets
    mt_med = subset_pick["MT"].median()
    rep_row = subset_pick.iloc[(subset_pick["MT"] - mt_med).abs().argmin()]
    ep = int(rep_row["episode_id"])

    npz_path = os.path.join(traj_dir, f"ep_{ep:04d}.npz")
    if not os.path.isfile(npz_path):
        continue
    data = np.load(npz_path)
    pos = np.asarray(data["position"], dtype=float)
    if pos.ndim != 2 or pos.shape[1] < 2 or len(pos) < 2:
        continue
    if TRAJ_TRIM_AUTORESET_TAIL and "start_pos" in data.files:
        sp = np.asarray(data["start_pos"], dtype=float).ravel()[:2]
        while len(pos) >= 2:
            last, prev = pos[-1, :2], pos[-2, :2]
            d_ls = float(np.linalg.norm(last - sp))
            d_ps = float(np.linalg.norm(prev - sp))
            jump = float(np.linalg.norm(last - prev))
            if jump >= 0.06 and d_ls <= 0.2 and d_ps > d_ls + 0.04:
                pos = pos[:-1]
            else:
                break
    if TRAJ_CLIP_AT_FIRST_HIT and "hit" in data.files:
        hit = np.asarray(data["hit"], dtype=bool)
        if hit.any():
            pos = pos[: int(np.argmax(hit)) + 1]
    if len(pos) < 2:
        continue

    c = f"C{int(tid) % 10}"

    # Stern am tatsächlichen Trajektorien-Ende (kein separater Marker am Soll-Ziel)
    # Genau ein Pfad fuer dieses Target: nur aufeinanderfolgende Schritte,
    # dadurch wird Start<->Ende niemals direkt verbunden.
    segments = np.stack([pos[:-1, :2], pos[1:, :2]], axis=1)
    lc = LineCollection(segments, colors=c, linewidths=1.6, alpha=0.9, zorder=4)
    ax.add_collection(lc)

    target_label = int(tid) + 1  # Anzeige 1..20 statt 0..19
    ax.plot([], [], "-", color=c, lw=1.6, label=f"Target {target_label}")
    ax.plot(pos[0, 0], pos[0, 1], "o", color=c, ms=5, zorder=5)   # Start
    ax.scatter(
        pos[-1, 0], pos[-1, 1], marker="*", s=160, c=c,
        edgecolors="black", linewidths=0.6, zorder=6,
    )
    # Groesster positiver Sprung in Schrittgeschwindigkeit (kein eigener Wert in npz)
    if TRAJ_MARK_ACCEL_SPIKE and len(pos) >= 4:
        d = np.diff(pos[:, :2], axis=0)
        speed = np.linalg.norm(d, axis=1) / STEP_DT
        d_speed = np.diff(speed)
        j = int(np.argmax(d_speed))
        if d_speed[j] > 0:
            ax.plot(
                pos[j + 1, 0], pos[j + 1, 1], "+", color=c,
                ms=12, mew=2.2, zorder=7, label="_nolegend_",
            )
    n_drawn += 1

ax.set_xlabel("x (norm)")
ax.set_ylabel("y (norm)")
# Ein allgemeiner Legenden-Eintrag fuer das Kreuz (nicht pro Target)
if TRAJ_MARK_ACCEL_SPIKE:
    ax.plot([], [], "+", color="black", ms=10, mew=2.0, label="moment of greatest\nacceleration")
# Standardisierte Achsen wie im normierten Joystick-Raum
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)

# Rechte Legenden-Box über volle Höhe
handles, labels = ax.get_legend_handles_labels()
ax_leg.set_xticks([])
ax_leg.set_yticks([])
for s in ax_leg.spines.values():
    s.set_visible(True)
ax_leg.legend(handles, labels, loc="upper left", fontsize=7, frameon=False)

plt.tight_layout()
_fig_traj = os.path.join(EVAL_DIR, "trajectories_overview.png")
plt.savefig(_fig_traj, dpi=150, bbox_inches="tight")
print(f"Gespeichert: {_fig_traj}")
plt.show()